<a href="https://colab.research.google.com/github/avinash-tiwary/ePic/blob/main/notebooks/03_1D_Landau_Damping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: 1D Collisionless Landau Damping

## 1. Theoretical Background
Lev Landau (1946) discovered that Langmuir electrostatic waves in a collisionless Maxwellian plasma damp exponentially through resonant wave-particle interaction:
$$\gamma_L = -\sqrt{\frac{\pi}{8}}\frac{\omega_{pe}}{(k\lambda_D)^3} \exp\left(-\frac{1}{2(k\lambda_D)^2} - \frac{3}{2}\right)$$
Particles moving slightly slower than the wave phase velocity $v_\phi = \omega / k$ absorb energy from the wave, while particles moving slightly faster give energy to the wave. Because a Maxwellian has $\partial f / \partial v < 0$ everywhere, there are always more slower particles than faster particles, causing net damping of wave energy.


In [ ]:
# ==============================================================
# Google Colab Setup & Package Installation
# ==============================================================
import sys
if 'google.colab' in sys.modules:
    print('Running in Google Colab. Installing ePic...')
    !git clone https://github.com/avinash-tiwary/ePic.git
    %cd ePic
    !pip install -e .
else:
    print('Running locally. Verifying ePic installation...')
    import epic
    print(f'ePic version {epic.__version__} loaded successfully!')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from epic.solvers.pic1d import PIC1DSolver

Nx = 256
k = 0.5
L = 2.0 * np.pi / k
N_particles = 120000
dt = 0.05
t_end = 25.0
v_th = 1.0
alpha = 0.05

k_ld = k * v_th
gamma_L_theory = -np.sqrt(np.pi / 8.0) / (k_ld**3) * np.exp(-0.5 / (k_ld**2) - 1.5)
print(f"Analytic Landau damping rate: gamma_L = {gamma_L_theory:.4f} omega_pe")

solver = PIC1DSolver(Nx=Nx, boxsize=L, dt=dt)
weight = (1.0 * L) / N_particles

# Invert cumulative density distribution n(x) = n0 * (1 + alpha*cos(k*x))
np.random.seed(42)
u = np.linspace(0.0, 1.0, N_particles, endpoint=False)
x = u * L
for _ in range(5):
    f = (x + (alpha / k) * np.sin(k * x)) / L - u
    df = (1.0 + alpha * np.cos(k * x)) / L
    x -= f / df
x = np.mod(x, L)

vel = np.random.normal(0.0, v_th, (N_particles, 3))
solver.add_species("thermal_electrons", q=-weight, m=weight, pos=x, vel=vel)
solver.initialize()

print(f"Simulating Landau Damping with N={N_particles} particles...")
solver.run(t_end=t_end)

time_arr = np.array(solver.history["time"])
max_e = np.array(solver.history["max_E"])

plt.figure(figsize=(9, 5), dpi=110)
plt.semilogy(time_arr, max_e, color="#0077bb", lw=2, label="PIC Measured $|E|_{max}(t)$")
plt.semilogy(time_arr, max_e[0] * np.exp(gamma_L_theory * time_arr), "r--", lw=2.5,
             label=rf"Linear Landau Theory ($\gamma_L = {gamma_L_theory:.3f}\,\omega_{{pe}}$)")
plt.xlabel(r"Time ($\omega_{pe} t$)")
plt.ylabel(r"Peak Electric Field $|E|_{max}$")
plt.title("Collisionless Landau Damping of Langmuir Wave")
plt.grid(True, linestyle="--", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 2. Animated Wave Damping & Phase-Mixing Movie

<p align="center"><img src="../docs/animations/landau_damping_1d.gif" width="80%" alt="Landau Damping Movie"/></p>